In [15]:
import kagglehub
import pandas as pd
# Download latest version
path = kagglehub.dataset_download("muhammadimran112233/employees-evaluation-for-promotion")

print("Path to dataset files:", path)
import os

df = pd.read_csv(os.path.join(path, "employee_promotion.csv"))
df.head()


Using Colab cache for faster access to the 'employees-evaluation-for-promotion' dataset.
Path to dataset files: /kaggle/input/employees-evaluation-for-promotion


,employee_id,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,awards_won,avg_training_score,is_promoted
0,65438,Sales & Marketing,region_7,Master's & above,f,sourcing,1,35,5.0,8,0,49.0,0
1,65141,Operations,region_22,Bachelor's,m,other,1,30,5.0,4,0,60.0,0
2,7513,Sales & Marketing,region_19,Bachelor's,m,sourcing,1,34,3.0,7,0,50.0,0
3,2542,Sales & Marketing,region_23,Bachelor's,m,other,2,39,1.0,10,0,50.0,0
4,48945,Technology,region_26,Bachelor's,m,other,1,45,3.0,2,0,73.0,0


In [16]:
# Điền 0 cho previous_year_rating của nhân viên có length_of_service == 1 trực tiếp trên df gốc
df.loc[df['length_of_service'] == 1, 'previous_year_rating'] = df.loc[df['length_of_service'] == 1, 'previous_year_rating'].fillna(0)
# Điền 0 cho bất kỳ giá trị NaN còn lại nào của previous_year_rating để đảm bảo sạch NaN
df['previous_year_rating'] = df['previous_year_rating'].fillna(0)

In [17]:
from sklearn.preprocessing import OrdinalEncoder

# 1. Tạo bản sao từ df gốc
df_hot = df_pr.copy()

# 2. Điền giá trị nan cho education trước để tránh lỗi
mode_edu = df_hot['education'].mode()[0]
df_hot['education'] = df_hot['education'].fillna(mode_edu)

# 3. Mã hóa Ordinal cho education (có thứ tự thiết lập sẵn)
ordinal_order = ["Below Secondary", "Bachelor's", "Master's & above"]
edu_encoder = OrdinalEncoder(categories=[ordinal_order])
df_hot['education'] = edu_encoder.fit_transform(df_hot[['education']])

# 4. Mã hóa Ordinal cho region (tự động theo bảng chữ cái để không sinh thêm 33 cột One-Hot)
region_encoder = OrdinalEncoder()
df_hot['region'] = region_encoder.fit_transform(df_hot[['region']])

# 5. Mã hóa One-hot cho các cột phân loại khác (đã loại bỏ 'region')
categorical_value = ['department', 'gender', 'recruitment_channel']
df_hot = pd.get_dummies(df_hot, columns=categorical_value, drop_first=True, dtype=int)

In [18]:
from sklearn.preprocessing import StandardScaler

df_sc = df_hot.copy()
scaler = StandardScaler()
# Chuẩn hóa các cột số
num_cols = ['no_of_trainings', 'age', 'previous_year_rating', 'length_of_service', 'avg_training_score']
df_sc[num_cols] = scaler.fit_transform(df_sc[num_cols])

# Loại bỏ cột employee_id định danh
df_sc.drop(columns='employee_id', inplace=True)
df_sc.head()

,region,education,no_of_trainings,age,previous_year_rating,length_of_service,awards_won,avg_training_score,is_promoted,department_Finance,department_HR,department_Legal,department_Operations,department_Procurement,department_R&D,department_Sales & Marketing,department_Technology,gender_m,recruitment_channel_referred,recruitment_channel_sourcing
0,31.0,2.0,-0.415276,0.025598,1.283878,0.500460,0,-1.041152,0,0,0,0,0,0,0,1,0,0,0,1
1,14.0,1.0,-0.415276,-0.627135,1.283878,-0.437395,0,-0.227276,0,0,0,0,1,0,0,0,0,1,0,0
2,10.0,1.0,-0.415276,-0.104948,-0.052623,0.265996,0,-0.967163,0,0,0,0,0,0,0,1,0,1,0,1
3,15.0,1.0,1.226063,0.547785,-1.389124,0.969387,0,-0.967163,0,0,0,0,0,0,0,1,0,1,0,0
4,18.0,1.0,-0.415276,1.331064,-0.052623,-0.906322,0,0.734578,0,0,0,0,0,0,0,0,1,1,0,0


In [19]:
df_sc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54808 entries, 0 to 54807
Data columns (total 20 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   region                        54808 non-null  float64
 1   education                     54808 non-null  float64
 2   no_of_trainings               54808 non-null  float64
 3   age                           54808 non-null  float64
 4   previous_year_rating          54808 non-null  float64
 5   length_of_service             54808 non-null  float64
 6   awards_won                    54808 non-null  int64  
 7   avg_training_score            54808 non-null  float64
 8   is_promoted                   54808 non-null  int64  
 9   department_Finance            54808 non-null  int64  
 10  department_HR                 54808 non-null  int64  
 11  department_Legal              54808 non-null  int64  
 12  department_Operations         54808 non-null  int64  
 13  d

In [20]:
from imblearn.over_sampling import SMOTE

# Tách X và y
X = df_sc.drop(columns=['is_promoted'])
y = df_sc['is_promoted']

# Thực hiện SMOTE cân bằng dữ liệu
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

print("Tỷ lệ lớp sau khi áp dụng SMOTE:")
print(y_res.value_counts())


Tỷ lệ lớp sau khi áp dụng SMOTE:
is_promoted
0    50140
1    50140
Name: count, dtype: int64


In [21]:
from sklearn.model_selection import train_test_split

# Chia Train-Test theo tỷ lệ 70-30 sau khi đã Scale và SMOTE
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.3, random_state=42, stratify=y_res)

print("Kích thước tập X_train:", X_train.shape)
print("Kích thước tập X_test:", X_test.shape)

Kích thước tập X_train: (70196, 19)
Kích thước tập X_test: (30084, 19)


In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score

# Thiết lập các siêu tham số cho Random Forest
param_distributions = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_base = RandomForestClassifier(random_state=42)

# Cấu hình RandomizedSearchCV với 5-fold cross validation
rf_random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_distributions,
    n_iter=15,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print("Bắt đầu huấn luyện mô hình...")
rf_random_search.fit(X_train, y_train)

print("\nSiêu tham số tốt nhất:")
print(rf_random_search.best_params_)

# Đánh giá mô hình tốt nhất trên tập Test
best_rf = rf_random_search.best_estimator_
y_pred = best_rf.predict(X_test)

print("\n--- BÁO CÁO ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST ---")
print(f"Độ chính xác (Accuracy Score): {accuracy_score(y_test, y_pred):.4f}")
print("\nBáo cáo chi tiết (Classification Report):")
print(classification_report(y_test, y_pred))

Bắt đầu huấn luyện mô hình...
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Siêu tham số tốt nhất:
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 30}

--- BÁO CÁO ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST ---
Độ chính xác (Accuracy Score): 0.9539

Báo cáo chi tiết (Classification Report):
              precision    recall  f1-score   support

           0       0.94      0.97      0.95     15042
           1       0.97      0.94      0.95     15042

    accuracy                           0.95     30084
   macro avg       0.95      0.95      0.95     30084
weighted avg       0.95      0.95      0.95     30084



In [23]:
import joblib

# 1. Lưu mô hình Random Forest tốt nhất
joblib.dump(best_rf, 'random_forest_model.pkl')

# 2. Lưu bộ chuẩn hóa StandardScaler (để dùng chuẩn hóa dữ liệu mới sau này)
joblib.dump(scaler, 'scaler.pkl')

print("Đã lưu mô hình và bộ scaler thành công!")

# --- Hướng dẫn cách load lại sau này để dự đoán: ---
# loaded_model = joblib.load('random_forest_model.pkl')
# loaded_scaler = joblib.load('scaler.pkl')
# 
# # Khi có dữ liệu mới X_new chưa scale:
# # X_new[num_cols] = loaded_scaler.transform(X_new[num_cols])
# # predictions = loaded_model.predict(X_new)


Đã lưu mô hình và bộ scaler thành công!
